# IsoAnnot merge

Merges IsoAnnot output for **annotated** (Ensembl 104) and **novel** (IsoQuant) transcripts into a
single table of per-transcript PFAM domain annotations.

Run this **on the IsoAnnot VM**, after both IsoAnnot runs have completed. See `README.md` for the
full pipeline and for how to produce the two `results_*` directories this notebook reads.

Output: `merged_annotations_all_git.csv`, consumed by `make_transcript_ann_for_portal_upload_depmapomics.ipynb`.

## Configuration

All run-specific paths live here. Update these to match the `--outputdir` values used for the two IsoAnnot runs.

In [ ]:
from pathlib import Path

# Root of the IsoAnnot checkout on the VM (the Dockerfile symlinks this to /home/ubuntu/IsoAnnot)
ISOANNOT_ROOT = Path("/home/ubuntu/IsoAnnot/IsoAnnot")

# --outputdir passed to ./isoannot.sh for each of the two runs. See README.md step 3
ENSEMBL_OUTDIR = ISOANNOT_ROOT / "results_ensembl104"
NOVEL_OUTDIR = ISOANNOT_ROOT / "results_novel_transcripts_v2"

# Derived input paths.
ENSEMBL_DIR = ENSEMBL_OUTDIR / "data" / "Hsapiens" / "config" / "ensembl"
ENSEMBL_CDNA_FASTA = ENSEMBL_DIR / "Homo_sapiens.GRCh38.cdna.all.fa"
ENSEMBL_PEP_FASTA = ENSEMBL_DIR / "Homo_sapiens.GRCh38.pep.all.fa"
ENSEMBL_SQANTI_CLASS = ENSEMBL_DIR / "sqanti_classification.txt"
ENSEMBL_SQANTI_NMD = ENSEMBL_DIR / "sqanti_NMD.txt"
ENSEMBL_GFF3_MOD = (
    ENSEMBL_OUTDIR / "data" / "Hsapiens" / "human_tappas_ensembl_annotation_file.gff3_mod"
)

NOVEL_DIR = NOVEL_OUTDIR / "data" / "Hsapiens" / "output" / "mytranscripts"
NOVEL_SQANTI_CLASS = NOVEL_DIR / "sqanti_classification.txt"
# The novel run names its NMD calls nmd_data.txt; the ensembl run uses sqanti_NMD.txt
NOVEL_NMD = NOVEL_DIR / "nmd_data.txt"
NOVEL_INTERPROSCAN_GTF = NOVEL_DIR / "layers" / "layer_interproscan.gtf"

OUTPUT_CSV = "merged_annotations_all_git.csv"

# Columns that must survive the intersection in the combine step below, because
# make_transcript_ann_for_portal_upload.ipynb requires them.
REQUIRED_OUTPUT_COLS = [
    "isoform",
    "chrom",
    "structural_category",
    "coding",
    "domain",
    "NMD_flag",
    "sequence",
]

for p in [
    ENSEMBL_CDNA_FASTA,
    ENSEMBL_PEP_FASTA,
    ENSEMBL_SQANTI_CLASS,
    ENSEMBL_SQANTI_NMD,
    ENSEMBL_GFF3_MOD,
    NOVEL_SQANTI_CLASS,
    NOVEL_NMD,
    NOVEL_INTERPROSCAN_GTF,
]:
    assert p.exists(), f"missing expected IsoAnnot output: {p}"
print("all expected IsoAnnot outputs found")

In [ ]:
import re

import pandas as pd

## Shared helper functions

In [ ]:
def parse_gff3_attributes(attr_text):
    """Pull ID / Name / Desc out of a GFF3-style attribute string."""
    text = "" if pd.isna(attr_text) else str(attr_text)

    def get_value(key):
        m = re.search(rf"(?:^|;)\s*{key}\s*=\s*([^;]*)", text)
        return m.group(1).strip() if m else None

    return {
        "ID": get_value("ID"),
        "Name": get_value("Name"),
        "Desc": get_value("Desc"),
    }


def pfam_domains_by_transcript(gtf_path):
    """Map transcript id -> set of PFAM domain names, from an IsoAnnot GFF3/GTF layer."""
    df = pd.read_csv(gtf_path, sep="\t", header=None)
    pfam = df[df[1].str.contains("PFAM") & (df[2] == "DOMAIN")]

    attrs = pfam[8].apply(parse_gff3_attributes).apply(pd.Series)
    with_attrs = pd.concat([pfam, attrs], axis=1)

    return with_attrs.groupby(with_attrs[0])["Name"].apply(lambda s: set(s.dropna()))

## Ensembl side

Build the protein-coding + translated-peptide filtered set, then attach PFAM domains from the
Ensembl `.gff3_mod`.

In [ ]:
def parse_cdna_header(header_line: str) -> dict:
    header = header_line[1:] if header_line.startswith(">") else header_line

    record = {
        "transcript_id": header.split()[0],
        "chromosome": None,
        "gene": None,
        "gene_biotype": None,
        "transcript_biotype": None,
        "gene_symbol": None,
        "Source": None,
        "HGNC": None,
    }

    for key in ["chromosome", "gene", "gene_biotype", "transcript_biotype", "gene_symbol"]:
        m = re.search(rf"\b{key}:([^\s]+)", header)
        if m:
            record[key] = m.group(1)

    source_match = re.search(r"\[Source:([^;\]]+)", header)
    if source_match:
        record["Source"] = source_match.group(1).strip().split()[0]

    hgnc_match = re.search(r"Acc:HGNC:(\d+)", header)
    if hgnc_match:
        record["HGNC"] = hgnc_match.group(1)

    return record


def parse_pep_header(header_line: str) -> dict:
    header = header_line[1:] if header_line.startswith(">") else header_line

    record = {
        "protein_id": header.split()[0],
        "chromosome": None,
        "gene": None,
        "transcript": None,
        "gene_biotype": None,
        "gene_symbol": None,
        "description": None,
        "Source": None,
        "HGNC": None,
    }

    for key in ["chromosome", "gene", "transcript", "gene_biotype", "gene_symbol"]:
        m = re.search(rf"\b{key}:([^\s]+)", header)
        if m:
            record[key] = m.group(1)

    description_match = re.search(r"\bdescription:(.*?)(?=\s+\[Source:|$)", header)
    if description_match:
        record["description"] = description_match.group(1).strip()

    source_match = re.search(r"\[Source:([^;\]]+)", header)
    if source_match:
        record["Source"] = source_match.group(1).strip().split()[0]

    hgnc_match = re.search(r"Acc:HGNC:(\d+)", header)
    if hgnc_match:
        record["HGNC"] = hgnc_match.group(1)

    return record


def read_fasta(fasta_path, header_parser):
    """Parse a FASTA into records, applying header_parser to each header line."""
    records = []
    current_header = None
    seq_chunks = []

    with open(fasta_path, "r") as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if current_header is not None:
                    row = header_parser(current_header)
                    row["sequence"] = "".join(seq_chunks)
                    records.append(row)
                current_header = line
                seq_chunks = []
            else:
                seq_chunks.append(line)

    if current_header is not None:
        row = header_parser(current_header)
        row["sequence"] = "".join(seq_chunks)
        records.append(row)

    return records

In [ ]:
cdna_df = pd.DataFrame(read_fasta(ENSEMBL_CDNA_FASTA, parse_cdna_header))
cdna_df = cdna_df[
    [
        "transcript_id",
        "chromosome",
        "gene",
        "gene_biotype",
        "transcript_biotype",
        "gene_symbol",
        "Source",
        "HGNC",
        "sequence",
    ]
]

pep_df = pd.DataFrame(read_fasta(ENSEMBL_PEP_FASTA, parse_pep_header))
pep_df = pep_df[["protein_id", "transcript", "sequence"]]

In [ ]:
merged_df = cdna_df.merge(pep_df, left_on="transcript_id", right_on="transcript", how="outer")
merged_df["isoform"] = merged_df["transcript_id"].str.split(".").str[0]

sqanti_df = pd.read_csv(ENSEMBL_SQANTI_CLASS, sep="\t")
sqanti_df = sqanti_df[
    [
        "isoform",
        "chrom",
        "strand",
        "length",
        "exons",
        "structural_category",
        "coding",
        "ORF_length",
        "CDS_length",
        "CDS_start",
        "CDS_end",
    ]
]
merged_df = merged_df.merge(sqanti_df, on="isoform", how="left")

sqanti_nmd = pd.read_csv(ENSEMBL_SQANTI_NMD, sep="\t", header=None)
sqanti_nmd.columns = ["isoform", "NMD_flag"]
merged_df = merged_df.merge(sqanti_nmd, on="isoform", how="left")

In [ ]:
# Filter to protein-coding transcripts that have a translated peptide.
# sequence_y is the peptide sequence from pep_df; sequence_x is the cDNA sequence.
merged_annotations = merged_df[
    (merged_df["gene_biotype"] == "protein_coding") & (merged_df["sequence_y"].notna())
].copy()
print(f"{len(merged_df)} -> {len(merged_annotations)} rows after protein-coding + peptide filter")

# The portal notebook joins UniProt on the *peptide* sequence, so expose sequence_y as `sequence`.
# (Previously this column arrived via the DeepTMHMM output, which echoed back the same peptides.)
merged_annotations = merged_annotations.rename(columns={"sequence_y": "sequence"})

In [ ]:
transcripts_with_domains = pfam_domains_by_transcript(ENSEMBL_GFF3_MOD)
merged_annotations = merged_annotations.merge(
    transcripts_with_domains, left_on="isoform", right_index=True, how="left"
)

## Novel side (`mytranscripts`)

Same shape as the Ensembl side: attach NMD calls, filter to transcripts with a predicted ORF, then
attach PFAM domains from `layer_interproscan.gtf`.

Note the NMD file is named `nmd_data.txt` here, not `sqanti_NMD.txt` as on the Ensembl side.

In [ ]:
sqanti_df_novel = pd.read_csv(NOVEL_SQANTI_CLASS, sep="\t")

nmd_annotation = pd.read_csv(NOVEL_NMD, sep="\t", header=None)
nmd_annotation.columns = ["isoform", "NMD_flag"]
sqanti_df_novel = sqanti_df_novel.merge(nmd_annotation, on="isoform", how="left")

sqanti_df_novel = sqanti_df_novel[
    [
        "isoform",
        "chrom",
        "associated_gene",
        "strand",
        "length",
        "exons",
        "structural_category",
        "coding",
        "ORF_length",
        "CDS_length",
        "CDS_start",
        "CDS_end",
        "ORF_seq",
        "NMD_flag",
    ]
]

In [ ]:
# Filter to novel transcripts with a predicted ORF.
merged_annotations_novel = sqanti_df_novel.dropna(subset=["ORF_seq"]).copy()
print(f"{len(sqanti_df_novel)} -> {len(merged_annotations_novel)} rows after ORF filter")

# The novel equivalent of the Ensembl peptide sequence is SQANTI's predicted ORF. Expose it under
# the same name so both sides carry `sequence` into the UniProt join. Note these are *predicted*
# ORFs, so few will match Swiss-Prot exactly.
merged_annotations_novel = merged_annotations_novel.rename(columns={"ORF_seq": "sequence"})

In [ ]:
transcripts_with_domains_novel = pfam_domains_by_transcript(NOVEL_INTERPROSCAN_GTF)
merged_annotations_novel = merged_annotations_novel.merge(
    transcripts_with_domains_novel, left_on="isoform", right_index=True, how="left"
)

## Combine Ensembl + novel

Keep only columns present on both sides, add the `chr` prefix to `chrom`, concat, and rename
`Name` -> `domain`.

The intersection silently drops anything missing from one side, so the cell below asserts that
every column in `REQUIRED_OUTPUT_COLS` survived. Both `sequence` (Ensembl peptide / novel predicted
ORF) and `NMD_flag` reach the output only because each side sets them up explicitly above.

In [ ]:
intersection_cols = merged_annotations_novel.columns.intersection(merged_annotations.columns)

merged_annotations_novel_final = merged_annotations_novel[intersection_cols].copy()
merged_annotations_ensembl_final = merged_annotations[intersection_cols].copy()

merged_annotations_novel_final["chrom"] = "chr" + merged_annotations_novel_final["chrom"].astype(str)
merged_annotations_ensembl_final["chrom"] = (
    "chr" + merged_annotations_ensembl_final["chrom"].astype(str)
)

merged_annotations_all = pd.concat(
    [merged_annotations_ensembl_final, merged_annotations_novel_final], ignore_index=True
)
merged_annotations_all = merged_annotations_all.rename(columns={"Name": "domain"})

# The intersection silently drops any column missing from either side. Fail loudly instead,
# so a column the portal notebook needs can't go missing unnoticed.
missing = [c for c in REQUIRED_OUTPUT_COLS if c not in merged_annotations_all.columns]
assert not missing, (
    f"columns dropped by the intersection: {missing}. "
    f"ensembl-only: {sorted(set(merged_annotations.columns) - set(intersection_cols))}; "
    f"novel-only: {sorted(set(merged_annotations_novel.columns) - set(intersection_cols))}"
)

print(f"{len(merged_annotations_all)} combined rows; columns: {list(merged_annotations_all.columns)}")
print(merged_annotations_all["NMD_flag"].value_counts(dropna=False))

In [ ]:
merged_annotations_all.to_csv(OUTPUT_CSV, index=False)
print(f"wrote {OUTPUT_CSV}")